In [1]:
import nest_asyncio

nest_asyncio.apply()

In [12]:
import sys
sys.path.append("..")
from src.helper import API_KEY, PDFPlumberReader

pass

In [3]:
from llama_index.core.tools import FunctionTool

def add(x: int, y: int) -> int:
    """Cộng hai số tự nhiên"""
    return x + y

def mystery(x: int, y: int) -> int:
    """Bình phương tổng 2 số tự nhiên"""
    return (x + y) ** 2

add_tool = FunctionTool.from_defaults(fn=add)
mystery_tool = FunctionTool.from_defaults(fn=mystery)

In [11]:
from llama_index.core.callbacks import base_handler
from llama_index.llms.groq import Groq
from llama_index.core.agent import FunctionAgent
from llama_index.core.agent import AgentWorkflow
from llama_index.core.agent.workflow import AgentStream, ToolCallResult
import asyncio

llm = Groq(
    model="openai/gpt-oss-20b",
    api_key=API_KEY.GROQ_API_KEY,
)

agent = FunctionAgent(
    name="react_assistant",
    tools=[add_tool, mystery_tool],
    llm=llm,
    system_prompt="you are a good assistant"
)

workflow = AgentWorkflow(
    agents=[agent],
    root_agent="react_assistant"
)

async def test():
    handler = workflow.run(
        user_msg="tell me output of the mystery function on 2 and 3"
    )
    async for ev in handler.stream_events():
        if isinstance(ev, ToolCallResult):
            print(f"\nCall {ev.tool_name} with {ev.tool_kwargs}\nReturned: {ev.tool_output}")
        if isinstance(ev, AgentStream):
            print(ev.delta, end="", flush=True)
    
    await handler

In [24]:
from llama_index.core import SimpleDirectoryReader

reader = SimpleDirectoryReader(
    input_files=["../data/metagpt.pdf"],
    file_extractor={".pdf":PDFPlumberReader()}   
)

documents = reader.load_data()

In [30]:
from llama_index.core.node_parser import SentenceSplitter

splitter = SentenceSplitter(
    chunk_size=1024,
    chunk_overlap=250
)
nodes = splitter.get_nodes_from_documents(
    documents,
    show_progress=True
)

Parsing nodes: 100%|██████████| 29/29 [00:00<00:00, 790.35it/s]


In [56]:
print(nodes[0].metadata)

{'page': 1}


In [37]:
from llama_index.core import Settings
from llama_index.embeddings.nvidia import NVIDIAEmbedding

Settings.llm = llm

Settings.embed_model = NVIDIAEmbedding(
    nvidia_api_key=API_KEY.NVIDIA_API_KEY,
    model="nvidia/nemotron-3-embed-1b"
)

d:\Building Agentic RAG with Llamaindex\.venv\Lib\site-packages\llama_index\embeddings\nvidia\base.py:206: UserWarning: Unable to determine validity of nvidia/nemotron-3-embed-1b
  warnings.warn(f"Unable to determine validity of {model_name}")


In [ ]:
from llama_index.core import VectorStoreIndex

vector_index = VectorStoreIndex(nodes, show_progress=True)

Generating embeddings: 100%|██████████| 35/35 [00:05<00:00,  6.75it/s]


In [59]:
from llama_index.core.vector_stores import MetadataFilter, MetadataFilters, FilterCondition

def vector_query(query: str, page_numbers: list[int]) -> str:
    """Perform a vector search over an index.
    query (str): the string query to be embedded.
    page_numbers (list[str]): Filter by set of pages. Leave BLANK if we want to perform a vector search over all pages. Otherwise, filter by the set of specified pages.
    """
    metadata_dicts = [MetadataFilter(key="page", value=int(p)) for p in page_numbers]
    vector_query_engine = vector_index.as_query_engine(
        similarity_top_k=2,
        filters=MetadataFilters.from_dicts(
            filter_dicts=metadata_dicts,
            condition=FilterCondition.OR
        )
    )
    return vector_query_engine.query(query)

vector_query_tool = FunctionTool.from_defaults(
    name="vector_tool",
    fn=vector_query
)

In [60]:
response = llm.predict_and_call(
    tools=[vector_query_tool],
    user_msg="What are the high-level results of MetaGPT as described on page 2?",
    verbose=True
)
print(str(response))

=== Calling Function ===
Calling function: vector_tool with args: {"page_numbers": [2], "query": "MetaGPT high-level results page 2"}
=== Function Output ===
**MetaGPT High‑Level Results (Page 2)**  

- **Code‑generation performance**  
  - Pass@1 on the HumanEval benchmark: **85.9 %**  
  - Pass@1 on the MBPP benchmark: **87.7 %**  

- **Task completion**  
  - Achieved a **100 % task‑completion rate** across all evaluated projects, demonstrating full robustness and efficiency.  

- **Comparative standing**  
  - Surpasses other popular frameworks (AutoGPT, LangChain, AgentVerse, ChatDev) in handling higher software complexity and delivering extensive functionality.  

- **Efficiency**  
  - Maintains strong time and token‑cost performance while executing automatic requirement analysis, system design, code generation, modification, execution, and debugging.
**MetaGPT High‑Level Results (Page 2)**  

- **Code‑generation performance**  
  - Pass@1 on the HumanEval benchmark: **85.9 %** 

In [ ]:
if __name__ == "__main__":
    await test() 